# Multi-Document Assistant using Hybrid Retrieval

This notebook demonstrates how to build a multi-document assistant using LangChain, Google Gemini, ChromaDB, and BM25 for hybrid retrieval. The assistant will answer questions based on a collection of PDF documents.

## 1. Setup and Installation

First, we install all the necessary Python libraries. This includes LangChain components, ChromaDB for vector storage, PyPDF for PDF loading, Rank BM25 for lexical search, and LangChain Google GenAI for interacting with Google's generative models and embeddings.

In [1]:
!pip install -q langchain langchain-community langchain-core chromadb pypdf rank_bm25 langchain-google-genai langchain-chroma

In [2]:
from langchain_community.document_loaders import PyPDFLoader

pdf_paths = [
    "/content/multi_document_assistant/Legal-Services-Agreement.pdf",
    "/content/multi_document_assistant/Contract_document.pdf"
]

documents = []

for path in pdf_paths:
    try:
        loader = PyPDFLoader(path)
        docs = loader.load()

        for doc in docs:
            doc.metadata["source"] = path.split("/")[-1]

        documents.extend(docs)
    except Exception as e:
        print(f"Error loading {path}: {e}")

print(f"Loaded {len(documents)} pages")

Loaded 38 pages


## 2. Document Loading and Preprocessing

We load the PDF documents using `PyPDFLoader`. Each page of the PDF is treated as a separate document, and we add the original filename as metadata.

### 2.1 Text Splitting

To manage context and improve retrieval accuracy, we split the loaded documents into smaller, overlapping chunks using `RecursiveCharacterTextSplitter`. This helps in breaking down large documents into manageable pieces while retaining context across splits.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=100
)

chunks = splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks")

Created 90 chunks


## 3. Vector Database and Hybrid Retrieval Setup

This section sets up the vector database using ChromaDB and defines the hybrid retrieval mechanism, combining vector similarity search with BM25 lexical search.

### 3.1 `create_vector_db` Function

This function handles the creation or loading of the Chroma vector database. It includes logic for batching document additions to the database, which can be useful for large datasets. It checks if an existing database is present to avoid recreating it.

In [4]:
import os
import time
from langchain_chroma import Chroma

def create_vector_db(chunks, embeddings, persist_dir="chroma_db"):
    if os.path.exists(persist_dir):
        print("Loading existing DB...")
        return Chroma(
            persist_directory=persist_dir,
            embedding_function=embeddings
        )

    print("Creating new DB (batched)...")

    texts = [doc.page_content for doc in chunks]
    metas = [doc.metadata for doc in chunks]

    vector_db = None
    batch_size = 10

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        batch_meta = metas[i:i+batch_size]

        print(f"Batch {i//batch_size + 1}")

        if vector_db is None:
            vector_db = Chroma.from_texts(
                texts=batch_texts,
                embedding=embeddings,
                metadatas=batch_meta,
                persist_directory=persist_dir
            )
        else:
            vector_db.add_texts(
                texts=batch_texts,
                metadatas=batch_meta
            )

        time.sleep(1)

    vector_db.persist()
    print("DB saved!")

    return vector_db

### 3.2 Initialize Embeddings and ChromaDB

We initialize the Google Generative AI embeddings model (`gemini-embedding-001`) and then use it to create or load our Chroma vector database from the pre-processed document chunks. The API key is securely fetched from Colab's secrets manager.

In [5]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from google.colab import userdata

# Fetch the API key from Colab's secrets manager
api_key = userdata.get('GEMINI_API_KEY')
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=api_key
)

vector_db = create_vector_db(
    chunks,
    embeddings
)

Loading existing DB...


### 3.3 Initialize BM25 Lexical Search

BM25 (Best Match 25) is a ranking function used in information retrieval that estimates the relevance of documents to a given search query. Here, we tokenize our document chunks to prepare them for BM25 indexing.

In [6]:
from rank_bm25 import BM25Okapi

chunk_texts = [doc.page_content for doc in chunks]
tokenized_corpus = [text.split() for text in chunk_texts]

bm25 = BM25Okapi(tokenized_corpus)

### 3.4 Hybrid Retrieval Function

The `hybrid_retrieve` function combines both vector similarity search (using the ChromaDB) and BM25 lexical search. It retrieves top documents from both methods and then combines them, removing duplicates, to provide a more comprehensive set of relevant documents. This leverages the strengths of both semantic and keyword-based search.

In [7]:
def hybrid_retrieve(query, k=3):
    # Vector search
    vector_results = vector_db.similarity_search(query, k=k)

    # BM25 search
    tokenized_query = query.split()
    bm25_scores = bm25.get_scores(tokenized_query)

    top_indices = sorted(
        range(len(bm25_scores)),
        key=lambda i: bm25_scores[i],
        reverse=True
    )[:k]

    bm25_results = [chunks[i] for i in top_indices]

    # Combine + remove duplicates
    combined = vector_results + bm25_results

    seen = set()
    final_docs = []

    for doc in combined:
        text = doc.page_content
        if text not in seen:
            seen.add(text)
            final_docs.append(doc)

    return final_docs[:k], vector_results, bm25_results

## 4. Language Model and Assistant Setup

This section configures the large language model (LLM) and defines the prompt structure for our legal assistant.

### 4.1 Initialize LLM and Prompt Template

We use `ChatGoogleGenerativeAI` with the `gemini-2.5-flash` model as our LLM. A `ChatPromptTemplate` is defined to instruct the LLM to act as a legal assistant, answering strictly based on the provided context from the documents.

In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    google_api_key=api_key
)

prompt = ChatPromptTemplate.from_template("""
You are a legal assistant.

Answer strictly based on the contract.

- Identify relevant clauses
- Highlight obligations or risks if present
- Be precise and professional
- If not found, say: Not found in document

Context:
{context}

Question:
{question}
""")

### 4.2 `ask_legal_assistant` Function

This function orchestrates the entire query process. It takes a user query, uses the `hybrid_retrieve` function to get relevant document chunks, formats them into a context, and then invokes the LLM with the context and the user's question. It returns the LLM's answer along with the retrieved documents for source attribution, and the separate vector and BM25 results.

In [9]:
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])


def ask_legal_assistant(query):
    docs, vector_res, bm25_res = hybrid_retrieve(query, k=3)
    context = format_docs(docs)

    response = llm.invoke(prompt.format(
        context=context,
        question=query
    ))

    return response.content, docs, vector_res, bm25_res

## 5. Ask a Question

Now, let's ask a question to our legal assistant and see its response, including the sources it used.

In [10]:
query = "What is the termination clause?"

answer, docs, vector_results,bm25_results  = ask_legal_assistant(query)

print("ANSWER:\n")
print(answer)

print("\nSOURCES:\n")

seen = set()
for doc in docs:
    source = doc.metadata.get("source", "Unknown")
    page = doc.metadata.get("page", "N/A")

    key = (source, page)
    if key not in seen:
        print(f"{source} (Page {page})")
        seen.add(key)

ANSWER:

**Relevant Clauses:**

*   **6.2 Termination:**
    *   "The Institute shall at any time be entitled to determine and terminate the contract for any reason including unsatisfactory performance or violation of Minimum Wages Act or of any of the other terms and conditions of the contract."
    *   "A notice in writing from the Institute to the service provider shall be issued giving 30 (Thirty) days time for such termination and vacation of the premises, without assigning any reasons thereof."
    *   "Under exceptional circumstances this 30 day period may be appropriately reduced."
    *   "If all or part of the contract is terminated in accordance with the provisions contained above, the Institute shall pay to the service provider charges up to the effective date of termination."
    *   "However, the termination of the contract shall not relieve the service provider of any of his obligations imposed by the contract with respect to the work performed by them prior to such term

### 5.1 Compare Retrieval Results

To understand the contribution of each retrieval method, we can inspect the raw results from the vector search and BM25 search separately.

In [11]:
print("Vector Results:")
for doc in vector_results:
    print(doc.page_content[:100])

print("\nBM25 Results:")
for doc in bm25_results:
    print(doc.page_content[:100])

Vector Results:
provision of this agreement  is held in whole or in part to be unenfor ceable for any reason, the
re
be issued giving 30 (Thirty) days time for such termination and vacation of 
the premises, without a
from the service provider to the Institute.  
 
 The Security Deposit shall be refunded to the servi

BM25 Results:
6.2 Termination: 
 
 The Institute shall at any time be entitled to determine and terminate the 
con
to offer any messing facility  beyond the scope of the contract unless  specific 
prior written appr
24 
made good under the contract and not paid for. Any deficiency shall 
forthwith be made good and 
